In [70]:
from pyscf import gto, scf, cc

a = 3 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 2 # frozen orbital per monomer
elmt = 'O'
basis = 'sto6g'
# for nc in nc_list:
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis=basis, unit='B', spin=0, verbose=4)
mol.build()

mf = scf.UHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]

et = mycc.ccsd_t()
print(et)

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Wed Mar  4 21:28:31 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA   

In [71]:
# example for PT2

options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 1,
            'dt':0.005,
            'seed': 2,
            'walker_type': 'uhf',
            'trial': 'ustoccsd2',
            'nslater': 1,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted.mixed_wave import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (8, 8)
# Number of basis functions: 10
# Number of Cholesky vectors: 36
#


In [72]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
prop_data["e_estimate"] = e_init
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# Decomposing Unrestricted T2 amplitudes
# Throw 4 vectors in T2 deomposition
# SVD cutoff = 1.00e-08 | error = 1.57e-09
# number of T2 decomposition vectors 28
# nelec: (8, 8)
# norb: 10
# nchol: 36
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 1
# dt: 0.005
# seed: 2
# walker_type: uhf
# trial: ustoccsd2
# nslater: 1
# use_gpu: False
# n_exp_terms: 6
# n_batch: 1
# max_error: 0.001
-149.03138634633396
4.308360848881421e-05


In [73]:
(taua, taub) = trial.decompose_t2((wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']), thresh=1e-8)
print(taua.shape, taub.shape)

# Decomposing Unrestricted T2 amplitudes
# Throw 4 vectors in T2 deomposition
# SVD cutoff = 1.00e-08 | error = 1.57e-09
# number of T2 decomposition vectors 28
(28, 8, 2) (28, 8, 2)


In [74]:
t2aa_rec = jnp.einsum('gia,gjb->iajb', taua, taua)
t2ab_rec = jnp.einsum('gia,gjb->iajb', taua, taub)
t2bb_rec = jnp.einsum('gia,gjb->iajb', taub, taub)

In [75]:
print(jnp.abs(t2aa_rec*2 - wave_data['t2aa']).max())
print(jnp.abs(t2ab_rec - wave_data['t2ab']).max())
print(jnp.abs(t2bb_rec*2 - wave_data['t2bb']).max())

1.285603992284723e-13
7.75226891838918e-10
1.2856131116734384e-13


In [76]:
# energy evaluator for <exp(T1)HF|(xtau + 1/2(xtau)^2)H|walker>
# exploitate the speed up by disconnected doubles

from jax import jit, lax, jvp
import opt_einsum as oe

# ad reference 
@jit
def _green(walker_up: jax.Array, 
           walker_dn: jax.Array, 
           slater_up: jax.Array,
           slater_dn: jax.Array
           ):
    '''
    full green's function 
    <psi|a_p^dagger a_q|walker>/<exp(T1)HF|walker>
    '''
    green_a = (walker_up @ (jnp.linalg.inv(slater_up.T.conj() @ walker_up)) @ slater_up.T.conj()).T
    green_b = (walker_dn @ (jnp.linalg.inv(slater_dn.T.conj() @ walker_dn)) @ slater_dn.T.conj()).T
    return [green_a, green_b]

@partial(jit, static_argnums=0)
def _walker_olp(
    trial,
    walker_up: jax.Array, 
    walker_dn: jax.Array, 
    slater_up: jax.Array,
    slater_dn: jax.Array
                ) -> complex:
    ''' 
    <psi|walker>
    '''
    olp = jnp.linalg.det(slater_up.T.conj() @ walker_up) * \
            jnp.linalg.det(slater_dn.T.conj() @ walker_dn)
    return olp

# @partial(jit, static_argnums=0)
# def _ci_walker_olp(trial,
#                    walker_up: jax.Array, 
#                    walker_dn: jax.Array, 
#                    slater_up: jax.Array,
#                    slater_dn: jax.Array,
#                    ci1, ci2) -> complex:
#     ''' 
#     unrestricted cisd walker overlap
#     <(1+ci1+ci2)psi|walker>
#     = (cA_ia* + cB_ia*) <psi|i+a|walker> 
#     + 1/2 (cAA_iajb* + 2 cAB_iajb* + cBB_iajb*) <psi|i+j+ab|walker>
#     '''
#     c1a, c1b = ci1
#     c2aa, c2ab, c2bb = ci2
#     c1a = c1a.conj()
#     c1b = c1b.conj()
#     c2aa = c2aa.conj()
#     c2ab = c2ab.conj()
#     c2bb = c2bb.conj()
#     nocca, noccb = trial.nelec
#     norb = trial.norb
#     greena, greenb = _green(walker_up, walker_dn, slater_up, slater_dn)
#     greena_ov = greena[:nocca, nocca:]
#     greenb_ov = greenb[:noccb, noccb:]
#     o0 = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
#     o1 = oe.contract("ia,ia", c1a, greena_ov, backend="jax") \
#         + oe.contract("ia,ia", c1b, greenb_ov, backend="jax")
#     o2 = oe.contract("iajb, ia, jb", c2aa, greena_ov, greena_ov, backend="jax") \
#         + oe.contract("iajb, ia, jb", c2bb, greenb_ov, greenb_ov, backend="jax") \
#         + oe.contract("iajb, ia, jb", c2ab, greena_ov, greenb_ov, backend="jax")
#     return (1.0 + o1 + o2) * o0

@partial(jit, static_argnums=0)
def _ci_walker_olp_dc(trial,
                      walker_up: jax.Array,
                      walker_dn: jax.Array, 
                      slater_up: jax.Array, 
                      slater_dn: jax.Array,
                      ci1) -> complex:
    ''' 
    <(1+ci1+ci2)psi|walker> for disconnected doubles
    = (cA + cB) <psi|ia|walker> + 1/2 (cAcA + cAcB + cBcA + cBcB) <psi|i+j+ab|walker>
    '''
    c1a, c1b = ci1
    c1a = c1a.conj()
    c1b = c1b.conj()
    nocca = walker_up.shape[1]
    noccb = walker_dn.shape[1]
    greena, greenb = _green(walker_up, walker_dn, slater_up, slater_dn)
    greena_ov = greena[:nocca, nocca:]
    greenb_ov = greenb[:noccb, noccb:]
    ciga = oe.contract('ia,ja->ij', c1a, greena_ov, backend='jax')
    cigb = oe.contract('ia,ja->ij', c1b, greenb_ov, backend='jax')
    o0 = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
    o1a = oe.contract("ii->", ciga, backend="jax")
    o1b = oe.contract("ii->", cigb, backend="jax")
    o1 = o1a + o1b
    o2_c = o1**2 / 2
    o2_e = -(oe.contract("ij,ji->", ciga, ciga, backend="jax")
             +oe.contract("ij,ji->", cigb, cigb, backend="jax")) / 2
    o2 = o2_c + o2_e
    return (1.0 + o1 + o2) * o0

@partial(jit, static_argnums=0)
def _exp_h1(trial,
               x,
               h1_mod, 
               walker_up: jax.Array, 
               walker_dn: jax.Array,
               slater_up: jax.Array, 
               slater_dn: jax.Array, 
               ci1, ci2
               ) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h1_mod)|walker>
    '''

    walker_up_1x = walker_up + (x * h1_mod[0]) @ walker_up
    walker_dn_1x = walker_dn + (x * h1_mod[1]) @ walker_dn
    o_exp = _ci_walker_olp_dc(trial, walker_up_1x, walker_dn_1x, slater_up, slater_dn, ci1)

    return o_exp 

@partial(jit, static_argnums=0)
def _exp_h2(trial, 
               x, 
               chol_i, 
               walker_up: jax.Array,
               walker_dn: jax.Array,
               slater_up: jax.Array,
               slater_dn: jax.Array,
               ci1, ci2
               ) -> complex:
    '''
    <exp(T1)HF|(1+ci1+ci2) exp(x*h2)|walker>
    '''
    walker_up_2x = (
        walker_up
        + x * chol_i[0].dot(walker_up)
        + x**2 / 2.0 * chol_i[0].dot(chol_i[0].dot(walker_up))
    )
    walker_dn_2x = (
        walker_dn
        + x * chol_i[1].dot(walker_dn)
        + x**2 / 2.0 * chol_i[1].dot(chol_i[1].dot(walker_dn))
    )
    o_exp = _ci_walker_olp_dc(trial, walker_up_2x, walker_dn_2x, slater_up, slater_dn, ci1)

    return o_exp

@partial(jit, static_argnums=0)
def _d2_exp_h2i(trial,
                   chol_i, 
                   walker_up: jax.Array,
                   walker_dn: jax.Array, 
                   slater_up: jax.Array,
                   slater_dn: jax.Array, 
                   ci1, ci2):
    x = 0.0
    f = lambda a: _exp_h2(trial, a, chol_i, walker_up, walker_dn, slater_up, slater_dn, ci1, ci2)
    _, d2f = jax.jvp(lambda x: jax.jvp(f, [x], [1.0])[1], [x], [1.0])
    return d2f


@partial(jit, static_argnums=0)
def _calc_energy_cisd_ad(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, wave_data, ci1, ci2):

    norb = trial.norb
    h0 = ham_data['h0']
    h1_mod = ham_data['h1_mod']
    chol = ham_data["chol"].reshape(2, -1, norb, norb)
    chol = chol.transpose(1,0,2,3)
    # slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']

    # one body
    f1 = lambda a: _exp_h1(trial, a, h1_mod, walker_up, walker_dn, slater_up, slater_dn, ci1, ci2)
    olp, d1_overlap = jvp(f1, [0.0], [1.0])

    # two body
    def scan_chol(carry, c):
        walker_up, walker_dn, slater_up, slater_dn = carry
        return carry, _d2_exp_h2i(trial, c, walker_up, walker_dn, slater_up, slater_dn, ci1, ci2)

    _, d2_overlap_i = lax.scan(scan_chol, (walker_up, walker_dn, slater_up, slater_dn), chol)
    d2_overlap = jnp.sum(d2_overlap_i)/2

    e12 = (d1_overlap + d2_overlap) / olp

    return olp, h0 + e12

In [ ]:
@partial(jit, static_argnums=0)
def _calc_energy_slater(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    slater_up: jax.Array,
    slater_dn: jax.Array,
    ham_data: dict,
    ) -> jax.Array:
    
    norb = trial.norb
    nocc_a, nocc_b = trial.nelec
    h0  = ham_data['h0']
    h1_a, h1_b = ham_data["h1"]
    chol_a = ham_data["chol"][0].reshape(-1, trial.norb, trial.norb)
    chol_b = ham_data["chol"][1].reshape(-1, trial.norb, trial.norb)
    green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn)
    
    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    e1 = hg_a + hg_b
 
    gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
    gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
    trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
    trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
    e2_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
    e2_2 = -(oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
               + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
    e2 = e2_1 + e2_2

    overlap = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
    energy = h0 + e1 + e2

    return overlap, energy

# @partial(jit, static_argnums=0)
# def _calc_energy_cisd_dc(
#     trial, 
#     walker_up: jax.Array,
#     walker_dn: jax.Array,
#     slater_up: jax.Array,
#     slater_dn: jax.Array,
#     ham_data: dict, 
#     # wave_data: dict,
#     ci1,
#     ):

#     '''
#     Disconnected Doubles!!!
#     <(1+ci1+ci2)psi|H|walker>
#     = (cA + cB) <psi|ia H|walker> + 1/2 (cAcA + cAcB + cBcA + cBcB) <psi|i+j+ab H|walker>
#     A local energy evaluator for <(1+C1+C2)psi|H|walker> / <(1+C1+C2)psi|walker>
#     all operators and the walkers and psi are in the same basis (normally MO)
#     |psi> is not necesarily diagonal
    
#     all green's function and the chol and ci coeff are as their original definition
#     no half rotation performed
#     '''
#     norb = trial.norb
#     nocc_a, nocc_b = trial.nelec
#     h0  = ham_data['h0']
#     h1_a, h1_b = ham_data["h1"]
#     # slater_up, slater_dn = wave_data["mo_ta"], wave_data["mo_tb"]
#     chol_a = ham_data["chol"][0].reshape(-1, trial.norb, trial.norb)
#     chol_b = ham_data["chol"][1].reshape(-1, trial.norb, trial.norb)
#     green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn) # full green
#     greenov_a = green_a[:nocc_a, nocc_a:]
#     greenov_b = green_b[:nocc_b, nocc_b:]
#     greenp_a = (green_a - jnp.eye(trial.norb))[:, nocc_a:]
#     greenp_b = (green_b - jnp.eye(trial.norb))[:, nocc_b:]
    
#     # applied to the bra
#     c1_a, c1_b = ci1
#     c1_a = c1_a.conj()
#     c1_b = c1_b.conj()

#     ######################## universal terms #########################
#     c1g_a = oe.contract("ia,ja->ij", c1_a, greenov_a, backend="jax")
#     c1g_b = oe.contract("ia,ja->ij", c1_b, greenov_b, backend="jax")
#     c1gp_a = oe.contract("ia,pa->ip", c1_a, greenp_a, backend="jax")
#     c1gp_b = oe.contract("ia,pa->ip", c1_b, greenp_b, backend="jax")
#     c1gg_a = oe.contract("ij,iq->jq", c1g_a, green_a[:nocc_a,:], backend="jax") # c_ia G_ja G_iq
#     c1gg_b = oe.contract("ij,iq->jq", c1g_b, green_b[:nocc_b,:], backend="jax")
#     c1gpg_a = oe.contract("ip,iq->pq", c1gp_a, green_a[:nocc_a,:], backend="jax") # c_ia Gp_pa G_iq
#     c1gpg_b = oe.contract("ip,iq->pq", c1gp_b, green_b[:nocc_b,:], backend="jax")
    
#     ########################## overlap terms #########################
#     o0 = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
#     o1_a = oe.contract("ii->", c1g_a, backend="jax")
#     o1_b = oe.contract("ii->", c1g_b, backend="jax")
#     o1 = o1_a + o1_b
#     o2_c = o1**2 / 2
#     o2_e = -(oe.contract("ij,ji->", c1g_a, c1g_a, backend="jax")
#              +oe.contract("ij,ji->", c1g_b, c1g_b, backend="jax")) / 2
#     o2 = o2_c + o2_e
#     overlap =  (1.0 + o1 + o2) * o0

#     ########################### ref energy ############################
#     gh_a = oe.contract("pr,qr->pq", green_a, h1_a, backend="jax")
#     gh_b = oe.contract("pr,qr->pq", green_b, h1_b, backend="jax")
#     trgh_a = oe.contract("pp->", gh_a, backend="jax")
#     trgh_b = oe.contract("pp->", gh_b, backend="jax")
#     e1_0 = trgh_a + trgh_b
 
#     gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
#     gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
#     trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
#     trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
#     e2_0_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
#     e2_0_2 = -(oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
#                + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
#     e2_0 = e2_0_1 + e2_0_2

#     ############################ ci terms #############################

#     ###### one-body single excitations ######
#     e1_1_1 = o1 * e1_0

#     e1_1_2 = -(oe.contract("pq,pq->", c1gpg_a, h1_a, backend="jax")
#                + oe.contract("pq,pq->", c1gpg_b, h1_b, backend="jax"))
    
#     e1_1 = e1_1_1 + e1_1_2 # <C1 psi|h1|walker>/<psi|walker>

#     ###### one-body double excitations ######
#     e1_2_1 = o2 * e1_0

#     c2ggg_aaa_c = o1_a * c1gpg_a / 2 # cA_ia cA_jb GA_ia GA_jq GpA_pb (-)
#     c2ggg_aaa_e = oe.contract('jp,jq->pq', c1gp_a, c1gg_a, backend='jax') / 2 # cA_ia cA_jb GA_ja GA_iq GpA_pb (+)
#     c2ggg_aaa = 2 * (c2ggg_aaa_c - c2ggg_aaa_e) # swap ia, jb pairwise
#     c2ggg_aba = o1_b * c1gpg_a / 2 # cB_jb GB_jb  cA_ia GpA_pa  GA_iq
#     c2ggg_baa =  c2ggg_aba # cB_ia GB_ia  cA_jb GpA_pb  GA_jq
#     c2ggg_bbb_c = o1_b * c1gpg_b / 2
#     c2ggg_bbb_e = oe.contract('jp,jq->pq', c1gp_b, c1gg_b, backend='jax') / 2
#     c2ggg_bbb = 2 * (c2ggg_bbb_c - c2ggg_bbb_e)
#     c2ggg_bab = o1_a * c1gpg_b
#     c2ggg_abb = c2ggg_bab
#     e1_2_2_a = -oe.contract("pq,pq->", c2ggg_aaa + c2ggg_aba + c2ggg_baa, h1_a, backend="jax")
#     e1_2_2_b = -oe.contract("pq,pq->", c2ggg_bbb + c2ggg_bab + c2ggg_abb, h1_b, backend="jax")
#     e1_2_2 = e1_2_2_a + e1_2_2_b
#     e1_2 = e1_2_1 + e1_2_2  # <C2 psi|h1|walker>/<psi|walker>

#     ###### two-body single excitations ######
#     e2_1_1 = o1 * e2_0

#     # c_ia Gp_pa G_ir L_pr G_qs L_qs
#     lc1g_a = oe.contract("gpq,pq->g", chol_a, c1gpg_a, backend="jax")
#     lc1g_b = oe.contract("gpq,pq->g", chol_b, c1gpg_b, backend="jax")
#     e2_1_2 = -((lc1g_a + lc1g_b) @ (trgl_a + trgl_b))

#     glc1gp_a = jnp.einsum("gpq,iq->gpi", gl_a, c1gp_a, optimize="optimal") # t_ia Gp_qa G_pr L_qr 
#     glc1gp_b = jnp.einsum("gpq,iq->gpi", gl_b, c1gp_b, optimize="optimal")
#     e2_1_3 = jnp.einsum("gpi,gip->", glc1gp_a, gl_a[:,:nocc_a,:], optimize="optimal") \
#             + jnp.einsum("gpi,gip->", glc1gp_b, gl_b[:,:nocc_b,:], optimize="optimal") # t_ia Gp_pa G_qr L_pr G_is L_qs
    
#     e2_1 = e2_1_1 + e2_1_2 + e2_1_3 # <C1 psi|h2|walker> / <psi|walker>

#     ###### two-body double excitations ######
#     e2_2_1 = o2 * e2_0

#     lt2g_a = oe.contract("gpq,pq->g", chol_a, c2ggg_aaa + c2ggg_aba + c2ggg_baa, backend="jax")
#     lt2g_b = oe.contract("gpq,pq->g", chol_b, c2ggg_bbb + c2ggg_bab + c2ggg_abb, backend="jax")
#     e2_2_2_1 = -((lt2g_a + lt2g_b) @ (trgl_a + trgl_b)) / 2

#     lc2ggg_a = oe.contract("gpr,qr->gpq", chol_a, c2ggg_aaa + c2ggg_aba + c2ggg_baa, backend="jax")
#     lc2ggg_b = oe.contract("gpr,qr->gpq", chol_b, c2ggg_aaa + c2ggg_bab + c2ggg_abb, backend="jax")
#     e2_2_2_2 = (oe.contract("gpq,gpq->", gl_a, lc2ggg_a, backend="jax")
#                 + oe.contract("gpq,gpq->", gl_b, lc2ggg_b, backend="jax")) / 2
#     glgp_a = oe.contract("giq,qa->gia", gl_a[:,:nocc_a,:], greenp_a, backend="jax")
#     glgp_b = oe.contract("giq,qa->gia", gl_b[:,:nocc_b,:], greenp_b, backend="jax")
#     l2t2_a = oe.contract("gia,ia->g", glgp_a, c1_a, backend="jax")
#     l2t2_b = oe.contract("gia,ia->g", glgp_b, c1_b, backend="jax")
#     e2_2_3 = jnp.sum((l2t2_a + l2t2_b)**2) / 2

#     e2_2_2 = e2_2_2_1 + e2_2_2_2
#     e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <C2 psi|h2|walker>/<psi|walker>

#     energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1 + o1 + o2)

#     return overlap, e1_0, e2_0, e1_1, e2_1, e1_2, e2_2

In [182]:
@partial(jit, static_argnums=0)
def _calc_energy_cisd(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    slater_up, 
    slater_dn,
    ham_data: dict,
    wave_data: dict,
    ci1, ci2,
) -> complex:
    
    '''
    c1 = c1a + c1b
    c2 = 1/4 c2aa + c2ab + 1/4 c2bb
       = 1/2 (1/2 c2aa + c2ab + c2ba + 1/2 c2bb)
    A local energy evaluator for <(1+C1+C2)psi|H|walker> / <(1+C1+C2)psi|walker>
    all operators and the walkers and psi are in the same basis (normally MO)
    |psi> is not necesarily diagonal
    
    all green's function and the chol and ci coeff are as their original definition
    no half rotation performed
    '''
    nocc_a = self.nelec[0]
    nocc_b = self.nelec[1]
    c1_a, c1_b = ci1
    c2_aa, c2_ab, c2_bb = ci2
    c1_a = c1_a.conj()
    c1_b = c1_b.conj()
    c2_aa = c2_aa.conj()
    c2_ab = c2_ab.conj()
    c2_bb = c2_bb.conj()
    
    # slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']
    h0 = ham_data["h0"]
    h1_a = ham_data["h1"][0]
    h1_b = ham_data["h1"][1]
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)

    # full green's function G_pq
    green_a, green_b = self._green(walker_up, walker_dn, slater_up, slater_dn)
    greenov_a = green_a[:nocc_a, nocc_a:]
    greenov_b = green_b[:nocc_b, nocc_b:]
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    ################## overlaps #########################
    o0 = self._slater_olp(walker_up, walker_dn, slater_up, slater_dn)
    o1 = oe.contract("ia,ia->", c1_a, greenov_a, backend="jax") \
        + oe.contract("ia,ia->", c1_b, greenov_b, backend="jax")
    o2 = 0.5 * oe.contract("iajb,ia,jb->", c2_aa, greenov_a, greenov_a, backend="jax") \
        + 0.5 * oe.contract("iajb,ia,jb->", c2_bb, greenov_b, greenov_b, backend="jax") \
        + oe.contract("iajb,ia,jb->", c2_ab, greenov_a, greenov_b, backend="jax")
    overlap =  (1.0 + o1 + o2) * o0

    ################## ref ###############################
    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    e1_0 = hg_a + hg_b # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>

    # two-body 
    gla = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
    glb = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
    trgla = oe.contract('gpp->g', gla, backend="jax")
    trglb = oe.contract('gpp->g', glb, backend="jax")
    e2_0_1 = 0.5 * jnp.sum((trgla + trglb)**2)
    e2_0_2 = - 0.5 * (oe.contract('gpq,gqp->', gla, gla, backend="jax")
                    + oe.contract('gpq,gqp->', glb, glb, backend="jax"))
    e2_0 = e2_0_1 + e2_0_2
    ########################################################

    # one body single excitations  <psi|T1 h1|walker>/<psi|HF|walker>
    e1_1_1 = o1 * e1_0

    gpc1_a = oe.contract("pa,ia->pi", greenp_a, c1_a, backend="jax") # greenp_a @ t1_a.T
    gpc1_b = oe.contract("pa,ia->pi", greenp_b, c1_b, backend="jax")
    c1_green_a = oe.contract("pi,iq->pq", gpc1_a, green_a[:nocc_a,:], backend="jax")
    c1_green_b = oe.contract("pi,iq->pq", gpc1_b, green_b[:nocc_b,:], backend="jax") # gpt1_b @ green_b
    e1_1_2 = -(oe.contract("pq,pq->", h1_a, c1_green_a, backend="jax")
            + oe.contract("pq,pq->", h1_b, c1_green_b, backend="jax"))
    
    e1_1 = e1_1_1 + e1_1_2 # <HF|T1 h1|walker>/<HF|walker>

    # one body double excitations  <psi|T2 h1|walker>/<psi|HF|walker>
    c2g_aa_a = oe.contract("iajb,ia->jb", c2_aa, greenov_a, backend="jax") / 4
    c2g_bb_b = oe.contract("iajb,ia->jb", c2_bb, greenov_b, backend="jax") / 4
    c2g_ab_a = oe.contract("iajb,jb->ia", c2_ab, greenov_b, backend="jax")
    c2g_ab_b = oe.contract("iajb,ia->jb", c2_ab, greenov_a, backend="jax")

    e1_2_1 = o2 * e1_0
    
    c2_ggg_aaa = (greenp_a @ c2g_aa_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    c2_ggg_aba = (greenp_a @ c2g_ab_a.T) @ green_a[:nocc_a,:]
    c2_ggg_bbb = (greenp_b @ c2g_bb_b.T) @ green_b[:nocc_b,:] 
    c2_ggg_bab = (greenp_b @ c2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract("pq,pq->", h1_a, 4 * c2_ggg_aaa + c2_ggg_aba, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, 4 * c2_ggg_bbb + c2_ggg_bab, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <psi|T2 h1|walker>/<psi|walker>

    # two body single excitations <psi|T1 h2|walker>/<psi|walker>
    e2_1_1 = o1 * e2_0

    # c_ia Gp_pa G_ir L_pr G_qs L_qs
    lc1g_a = oe.contract("gpq,pq->g", chol_a, c1_green_a, backend="jax")
    lc1g_b = oe.contract("gpq,pq->g", chol_b, c1_green_b, backend="jax")
    e2_1_2 = -((lc1g_a + lc1g_b) @ (trgla + trglb))

    # t_ia Gp_pa G_qr G_is L_pr L_qs
    c1gp_a = oe.contract("ia,pa->ip", c1_a, greenp_a, backend="jax") # t_ia Gp_pa 
    c1gp_b = oe.contract("ia,pa->ip", c1_b, greenp_b, backend="jax")
    glgpc1_a = jnp.einsum("gpq,iq->gpi", gla, c1gp_a, optimize="optimal") # t_ia Gp_pa G_qr L_pr
    glgpc1_b = jnp.einsum("gpq,iq->gpi", glb, c1gp_b, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gip->", glgpc1_a, gla[:,:nocc_a,:], optimize="optimal") \
            + jnp.einsum("gpi,gip->", glgpc1_b, glb[:,:nocc_b,:], optimize="optimal") # t_ia Gp_pa L_pr G_qr L_qs G_is
    
    e2_1 = e2_1_1 + e2_1_2 + e2_1_3 # <psi|ci1 h2|walker> / <psi|walker>

    # two body double excitations <psi|T2 h2|walker>/<psi|walker>
    e2_2_1 = o2 * e2_0

    lc2g_a = oe.contract("gpq,pq->g", chol_a, 8*c2_ggg_aaa + 2*c2_ggg_aba, backend="jax")
    lc2g_b = oe.contract("gpq,pq->g", chol_b, 8*c2_ggg_bbb + 2*c2_ggg_bab, backend="jax")
    e2_2_2_1 = -((lc2g_a + lc2g_b) @ (trgla + trglb)) / 2.0

    def scanned_fun(carry, x):
        chol_a_i, chol_b_i, gl_a_i, gl_b_i = x
        lc2_ggg_a_i = oe.contract("pr,qr->pq", chol_a_i, 8*c2_ggg_aaa + 2*c2_ggg_aba, backend="jax")
        lc2_ggg_b_i = oe.contract("pr,qr->pq", chol_b_i, 8*c2_ggg_bbb + 2*c2_ggg_bab, backend="jax")
        carry[0] += (oe.contract("pq,pq->", gl_a_i, lc2_ggg_a_i, backend="jax")
                    + oe.contract("pq,pq->", gl_b_i, lc2_ggg_b_i, backend="jax")) / 2 
        glgp_a_i = oe.contract("iq,qa->ia", gl_a_i[:nocc_a,:], greenp_a, backend="jax")
        glgp_b_i = oe.contract("iq,qa->ia", gl_b_i[:nocc_b,:], greenp_b, backend="jax")
        l2c2_aa = oe.contract("ia,jb,iajb->", 
                                glgp_a_i, # be carefull with single precision
                                glgp_a_i,
                                c2_aa, 
                                backend="jax") / 2
        l2c2_bb = oe.contract("ia,jb,iajb->", 
                                glgp_b_i, 
                                glgp_b_i, 
                                c2_bb, 
                                backend="jax") / 2
        l2c2_ab = oe.contract("ia,jb,iajb->", 
                                glgp_a_i, 
                                glgp_b_i, 
                                c2_ab, 
                                backend="jax")
        carry[1] += l2c2_aa + l2c2_ab + l2c2_bb
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0, 0.0], (chol_a, chol_b, gla, glb))

    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <psi|T2 h2|walker>/<psi|walker>

    energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1 + o1 + o2)
    return overlap, energy

In [205]:
@partial(jit, static_argnums=0)
def _calc_energy_cisd_dc(
    trial, 
    walker_up: jax.Array,
    walker_dn: jax.Array,
    slater_up: jax.Array,
    slater_dn: jax.Array,
    ham_data: dict, 
    # wave_data: dict,
    ci1,
    ):

    '''
    Disconnected Doubles!!!
    <(1+ci1+ci2)psi|H|walker>
    = (cA + cB) <psi|ia H|walker> + 1/2 (cAcA + cAcB + cBcA + cBcB) <psi|i+j+ab H|walker>
    A local energy evaluator for <(1+C1+C2)psi|H|walker> / <(1+C1+C2)psi|walker>
    all operators and the walkers and psi are in the same basis (normally MO)
    |psi> is not necesarily diagonal
    
    all green's function and the chol and ci coeff are as their original definition
    no half rotation performed
    '''
    norb = trial.norb
    nocc_a, nocc_b = trial.nelec
    h0  = ham_data['h0']
    h1_a, h1_b = ham_data["h1"]
    # slater_up, slater_dn = wave_data["mo_ta"], wave_data["mo_tb"]
    chol_a = ham_data["chol"][0].reshape(-1, trial.norb, trial.norb)
    chol_b = ham_data["chol"][1].reshape(-1, trial.norb, trial.norb)
    green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn) # full green
    greenov_a = green_a[:nocc_a, nocc_a:]
    greenov_b = green_b[:nocc_b, nocc_b:]
    greenp_a = (green_a - jnp.eye(trial.norb))[:, nocc_a:]
    greenp_b = (green_b - jnp.eye(trial.norb))[:, nocc_b:]
    
    # applied to the bra
    c1_a, c1_b = ci1
    c1_a = c1_a.conj()
    c1_b = c1_b.conj()

    ######################## universal terms #########################
    c1g_a = oe.contract("ia,ja->ij", c1_a, greenov_a, backend="jax")
    c1g_b = oe.contract("ia,ja->ij", c1_b, greenov_b, backend="jax")
    c1gp_a = oe.contract("ia,pa->ip", c1_a, greenp_a, backend="jax")
    c1gp_b = oe.contract("ia,pa->ip", c1_b, greenp_b, backend="jax")
    c1gg_a = oe.contract("ij,iq->jq", c1g_a, green_a[:nocc_a,:], backend="jax") # c_ia G_ja G_iq
    c1gg_b = oe.contract("ij,iq->jq", c1g_b, green_b[:nocc_b,:], backend="jax")
    c1gpg_a = oe.contract("ip,iq->pq", c1gp_a, green_a[:nocc_a,:], backend="jax") # c_ia Gp_pa G_iq
    c1gpg_b = oe.contract("ip,iq->pq", c1gp_b, green_b[:nocc_b,:], backend="jax")
    
    ########################## overlap terms #########################
    o0 = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
    o1_a = oe.contract("ii->", c1g_a, backend="jax")
    o1_b = oe.contract("ii->", c1g_b, backend="jax")
    o1 = o1_a + o1_b
    o2_c = o1**2 / 2
    o2_e = -(oe.contract("ij,ji->", c1g_a, c1g_a, backend="jax")
             +oe.contract("ij,ji->", c1g_b, c1g_b, backend="jax")) / 2
    o2 = o2_c + o2_e
    overlap =  (1.0 + o1 + o2) * o0

    ########################### ref energy ############################
    gh_a = oe.contract("pr,qr->pq", green_a, h1_a, backend="jax")
    gh_b = oe.contract("pr,qr->pq", green_b, h1_b, backend="jax")
    trgh_a = oe.contract("pp->", gh_a, backend="jax")
    trgh_b = oe.contract("pp->", gh_b, backend="jax")
    e1_0 = trgh_a + trgh_b
 
    gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
    gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
    trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
    trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
    e2_0_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
    e2_0_2 = -(oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
               + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
    e2_0 = e2_0_1 + e2_0_2

    ############################ ci terms #############################

    ###### one-body single excitations ######
    e1_1_1 = o1 * e1_0

    e1_1_2 = -(oe.contract("pq,pq->", c1gpg_a, h1_a, backend="jax")
               + oe.contract("pq,pq->", c1gpg_b, h1_b, backend="jax"))
    
    e1_1 = e1_1_1 + e1_1_2 # <C1 psi|h1|walker>/<psi|walker>

    ###### one-body double excitations ######
    e1_2_1 = o2 * e1_0

    c2ggg_aaa_c = o1_a * c1gpg_a # cA_ia cA_jb GA_ia GA_jq GpA_pb (-)
    c2ggg_aaa_e = oe.contract('jp,jq->pq', c1gp_a, c1gg_a, backend='jax') # cA_ia cA_jb GA_ja GA_iq GpA_pb (+)
    c2ggg_aaa = 2 * (c2ggg_aaa_c - c2ggg_aaa_e) # swap ia, jb pairwise
    c2ggg_aba = 2* o1_b * c1gpg_a # cB_jb GB_jb  cA_ia GpA_pa  GA_iq
    # c2ggg_baa = c2ggg_aba # cB_ia GB_ia  cA_jb GpA_pb  GA_jq
    c2ggg_bbb_c = o1_b * c1gpg_b
    c2ggg_bbb_e = oe.contract('jp,jq->pq', c1gp_b, c1gg_b, backend='jax')
    c2ggg_bbb = 2 * (c2ggg_bbb_c - c2ggg_bbb_e)
    c2ggg_bab = 2 * o1_a * c1gpg_b
    # c2ggg_abb = c2ggg_bab
    e1_2_2_a = -oe.contract("pq,pq->", c2ggg_aaa + c2ggg_aba, h1_a, backend="jax") / 2
    e1_2_2_b = -oe.contract("pq,pq->", c2ggg_bbb + c2ggg_bab, h1_b, backend="jax") / 2
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <C2 psi|h1|walker>/<psi|walker>

    ###### two-body single excitations ######
    e2_1_1 = o1 * e2_0

    # c_ia Gp_pa G_ir L_pr G_qs L_qs
    lc1g_a = oe.contract("gpq,pq->g", chol_a, c1gpg_a, backend="jax")
    lc1g_b = oe.contract("gpq,pq->g", chol_b, c1gpg_b, backend="jax")
    e2_1_2 = -jnp.sum((lc1g_a + lc1g_b) * (trgl_a + trgl_b))

    glc1gp_a = jnp.einsum("gpq,iq->gpi", gl_a, c1gp_a, optimize="optimal") # c_ia Gp_qa G_pr L_qr 
    glc1gp_b = jnp.einsum("gpq,iq->gpi", gl_b, c1gp_b, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gip->", glc1gp_a, gl_a[:,:nocc_a,:], optimize="optimal") \
            + jnp.einsum("gpi,gip->", glc1gp_b, gl_b[:,:nocc_b,:], optimize="optimal") # t_ia Gp_pa G_qr L_pr G_is L_qs
    
    e2_1 = e2_1_1 + e2_1_2 + e2_1_3 # <C1 psi|h2|walker> / <psi|walker>

    ###### two-body double excitations ######
    e2_2_1 = o2 * e2_0

    # ccGGGp G_qs L_ps
    lc2ggg_a = oe.contract("gpr,qr->gpq", chol_a, 2*(c2ggg_aaa + c2ggg_aba), backend="jax")
    lc2ggg_b = oe.contract("gpr,qr->gpq", chol_b, 2*(c2ggg_bbb + c2ggg_bab), backend="jax")
    trlc2ggg_a = oe.contract("gpp->g", lc2ggg_a, backend="jax")
    trlc2ggg_b = oe.contract("gpp->g", lc2ggg_b, backend="jax")
    e2_2_2_c = -jnp.sum((trlc2ggg_a + trlc2ggg_b)*(trgl_a + trgl_b)) / 4
    e2_2_2_e = (oe.contract("gpq,gpq->", gl_a, lc2ggg_a, backend="jax")
                + oe.contract("gpq,gpq->", gl_b, lc2ggg_b, backend="jax")) / 4
    e2_2_2 = e2_2_2_c + e2_2_2_e
    
    c1glgp_a = oe.contract("ip,gjp->gij", c1gp_a, gl_a[:,:nocc_a,:], backend="jax")
    c1glgp_b = oe.contract("ip,gjp->gij", c1gp_b, gl_b[:,:nocc_a,:], backend="jax")
    trc1glgp_a = oe.contract("gii->g", c1glgp_a, backend="jax")
    trc1glgp_b = oe.contract("gii->g", c1glgp_b, backend="jax")
    e2_2_3_c = jnp.sum((trc1glgp_a + trc1glgp_b)**2) / 2
    e2_2_3_e = (oe.contract("gij,gji->", c1glgp_a, c1glgp_a, backend="jax")
                + oe.contract("gij,gji->", c1glgp_b, c1glgp_b, backend="jax")) / 2
    e2_2_3 = e2_2_3_c - e2_2_3_e

    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <C2 psi|h2|walker>/<psi|walker>

    energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1 + o1 + o2)

    return overlap, energy

In [208]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
slater_up, slater_dn = prop_data['walkers'][0][0], prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape)+1j*np.random.rand(*walker_up.shape))/2
walker_dn = jnp.array(np.random.rand(*walker_up.shape)+1j*np.random.rand(*walker_up.shape))/2
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
# mo_ta, mo_tb = trial._thouless([prop_data['walkers'][0][0], prop_data['walkers'][0][0]], mycc.t1)
# ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
c1 = [jnp.array(np.random.rand(*mycc.t1[0].shape)+1j*np.random.rand(*mycc.t1[0].shape))/2, 
      jnp.array(np.random.rand(*mycc.t1[1].shape)+1j*np.random.rand(*mycc.t1[1].shape))/2]
c2aa = jnp.einsum("ia,jb->iajb", c1[0], c1[0]) * 2
c2ab = jnp.einsum("ia,jb->iajb", c1[0], c1[1])
c2bb = jnp.einsum("ia,jb->iajb", c1[1], c1[1]) * 2
c2aa = (c2aa - c2aa.transpose(0, 3, 2, 1)) / 2
c2bb = (c2bb - c2bb.transpose(0, 3, 2, 1)) / 2
c2 = [c2aa, c2ab, c2bb]
olp, energy = _calc_energy_cisd_ad(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, wave_data, c1, c2)
print(olp, energy)
overlap, energy = _calc_energy_cisd(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, wave_data, c1, c2)
print(overlap, energy)
overlap, energy = _calc_energy_cisd_dc(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, c1)
print(overlap, energy)

(1.1986272204492986e-06-3.2729490820520713e-06j) (-125.58274755887543-98.0927108076433j)
(1.1986272204492996e-06-3.27294908205206e-06j) (-125.58274755887562-98.09271080764368j)
(1.1986272204493022e-06-3.272949082052073e-06j) (-125.58274755887557-98.09271080764336j)


In [55]:
mycc.e_tot

np.float64(-0.9361048913679463)

In [30]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
t1 = [jnp.array(np.random.rand(*mycc.t1[0].shape)), jnp.array(np.random.rand(*mycc.t1[0].shape))]
c2aa = jnp.einsum("ia,jb->iajb", t1[0], t1[0]) * 0 
c2ab = jnp.einsum("ia,jb->iajb", t1[0], t1[1]) * 0 
c2bb = jnp.einsum("ia,jb->iajb", t1[1], t1[1]) * 0 
c2aa = (c2aa - c2aa.transpose(0, 3, 2, 1)) / 2
c2bb = (c2bb - c2bb.transpose(0, 3, 2, 1)) / 2
c2 = [c2aa, c2ab, c2bb]
mo_ta, mo_tb = trial._thouless([slater_up, slater_dn], t1)
olp, energy = _calc_energy_cisd_ad(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, wave_data, t1, c2)
print(olp, energy)
o_ex, e_ex = _calc_energy_slater(trial, walker_up, walker_dn, mo_ta, mo_tb, ham_data)
print(o_ex, e_ex)
olp, energy = _calc_energy_cisd_dc(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, t1)
print(olp, energy)

(1+0j) (-0.946109666425224+0j)
(1+0j) (-0.9461096664252244+0j)
(1+0j) (-0.9444189677889421+0j)


In [71]:
xtau_a = xtaus[0][0,2,:,:]
xtau_b = xtaus[1][0,2,:,:]
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
slater_up, slater_dn = trial._thouless((mo_ta, mo_tb), (xtau_a, xtau_b))
o_ex, e_ex = _calc_energy_slater(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data)
o_ci, e_ci = _calc_energy_cisd_dc(trial, walker_up, walker_dn, mo_ta, mo_tb, ham_data, (xtau_a, xtau_b))
print(o_ex, e_ex)
print(o_ci, e_ci)

(1+0j) (-0.9262825325618098+0.0035893881421029394j)
(1+0j) (-0.9411841806082923-0.055272863559906j)


In [74]:
t1 = [xtau_a, xtau_b]
c2aa = jnp.einsum("ia,jb->iajb", t1[0], t1[0])
c2ab = jnp.einsum("ia,jb->iajb", t1[0], t1[1])
c2bb = jnp.einsum("ia,jb->iajb", t1[1], t1[1])
c2aa = (c2aa - c2aa.transpose(0, 3, 2, 1)) / 2
c2bb = (c2bb - c2bb.transpose(0, 3, 2, 1)) / 2
c2 = [c2aa, c2ab, c2bb]
olp, energy = _calc_energy_ad(trial, walker_up, walker_dn, mo_ta, mo_tb, ham_data, wave_data, t1, c2)
print(olp, energy)

(1+0j) (-0.92628253256181+0.003589388142102967j)


In [61]:
mycc.e_tot

np.float64(-1.009485366963983)

In [ ]:
# slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']
c1 = (xtau_a*0, xtau_b)
c2aa = jnp.einsum('ia,jb->iajb', xtau_a, xtau_a)
c2ab = jnp.einsum('ia,jb->iajb', xtau_a, xtau_b)
c2bb = jnp.einsum('ia,jb->iajb', xtau_b, xtau_b)
# ci2aa = (c2aa - c2aa.transpose(0, 3, 2, 1)) / 2
# ci2bb = (c2bb - c2bb.transpose(0, 3, 2, 1)) / 2
c2 = (c2aa, c2ab, c2bb)
olp, energy = _calc_energy_ad(trial, walker_up, walker_dn, mo_ta, mo_tb, ham_data, wave_data, c1, c2)
print(olp, energy)

(1+0j) (-0.9556584131054568-0.004514307505845569j)


In [ ]:
# @jit
# def _ci_walker_olp_dc(walker_up: jax.Array,
#                       walker_dn: jax.Array, 
#                       slater_up: jax.Array, 
#                       slater_dn: jax.Array,
#                       ci1) -> complex:
#     ''' 
#     <(1+ci1+ci2)psi|walker> for disconnected doubles
#     = (cA + cB) <psi|ia|walker> + 1/2 (cAcA + cAcB + cBcA + cBcB) <psi|i+j+ab|walker>
#     '''
#     c1a, c1b = ci1
#     c1a = c1a.conj()
#     c1b = c1b.conj()
#     nocca = walker_up.shape[1]
#     noccb = walker_dn.shape[1]
#     greena, greenb = _green(walker_up, walker_dn, slater_up, slater_dn)
#     greena_ov = greena[:nocca, nocca:]
#     greenb_ov = greenb[:noccb, noccb:]
#     ciga = oe.contract('ia,ja->ij', c1a, greena_ov, backend='jax')
#     cigb = oe.contract('ia,ja->ij', c1b, greenb_ov, backend='jax')
#     o0 = _walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn)
#     o1a = oe.contract("ii->", ciga, backend="jax")
#     o1b = oe.contract("ii->", cigb, backend="jax")
#     o1 = o1a + o1b
#     o2_c = o1**2 / 2
#     o2_e = -(oe.contract("ij,ji->", ciga, ciga, backend="jax")
#              +oe.contract("ij,ji->", cigb, cigb, backend="jax")) / 2
#     o2 = o2_c + o2_e
#     return (1.0 + o1 + o2) * o0

In [96]:
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j*np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j*np.random.rand(*walker_dn.shape)) * 0.5
slater_up = jnp.array(np.random.rand(*walker_up.shape) + 1j*np.random.rand(*walker_up.shape)) * 0.5
slater_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j*np.random.rand(*walker_dn.shape)) * 0.5
ci1a = jnp.array(np.random.rand(nocca, nvira) + 1j*np.random.rand(nocca, nvira)) * 0.5
ci1b = jnp.array(np.random.rand(noccb, nvirb) + 1j*np.random.rand(noccb, nvirb)) * 0.5
ci2aa = jnp.einsum('ia,jb->iajb', ci1a, ci1a)
ci2ab = jnp.einsum('ia,jb->iajb', ci1a, ci1b)
ci2bb = jnp.einsum('ia,jb->iajb', ci1b, ci1b)
ci2aa = (ci2aa - ci2aa.transpose(0, 3, 2, 1)) / 2
ci2bb = (ci2bb - ci2bb.transpose(0, 3, 2, 1)) / 2
ci1 = [ci1a, ci1b]
ci2 = [ci2aa, ci2ab, ci2bb]
olp = _ci_walker_olp(trial, walker_up, walker_dn, slater_up, slater_dn, ci1, ci2)
print(olp)
olp_dc = _ci_walker_olp_dc(walker_up, walker_dn, slater_up, slater_dn, ci1)
print(olp_dc)

(0.06763381992702995-0.06466260043459209j)
(0.06763381992702995-0.06466260043459209j)


In [69]:
@partial(jit, static_argnums=0)
def _calc_energy_cid(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
    ci2,
) -> complex:
    nocc_a = self.nelec[0]
    nocc_b = self.nelec[1]
    c2_aa, c2_ab, c2_bb = ci2
    c2_aa = c2_aa.conj()
    c2_ab = c2_ab.conj()
    c2_bb = c2_bb.conj()

    h0 = ham_data['h0']
    h1_a, h1_b = ham_data["h1"]
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)
    slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']

    # full green's function G_pq
    green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn)
    greenov_a = green_a[:nocc_a, nocc_a:]
    greenov_b = green_b[:nocc_b, nocc_b:]
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    ################## overlaps #########################
    o0 = _walker_olp(walker_up, walker_dn, slater_up, slater_dn)
    o2 = 0.5 * oe.contract("iajb,ia,jb->", c2_aa, greenov_a, greenov_a, backend="jax") \
        + 0.5 * oe.contract("iajb,ia,jb->", c2_bb, greenov_b, greenov_b, backend="jax") \
        + oe.contract("iajb,ia,jb->", c2_ab, greenov_a, greenov_b, backend="jax")
    overlap =  (1.0 + o2) * o0

    ################## ref ###############################
    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    e1_0 = hg_a + hg_b

    gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
    gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
    trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
    trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
    e2_0_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
    e2_0_2 = - (oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
                + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
    e2_0 = e2_0_1 + e2_0_2
    ########################################################

    # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>
    # double excitations
    c2g_a = oe.contract("iajb,ia->jb", c2_aa, greenov_a, backend="jax") / 4
    c2g_b = oe.contract("iajb,ia->jb", c2_bb, greenov_b, backend="jax") / 4
    c2g_ab_a = oe.contract("iajb,jb->ia", c2_ab, greenov_b, backend="jax")
    c2g_ab_b = oe.contract("iajb,ia->jb", c2_ab, greenov_a, backend="jax")

    e1_2_1 = o2 * e1_0
    
    c2_ggg_aaa = (greenp_a @ c2g_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    c2_ggg_aba = (greenp_a @ c2g_ab_a.T) @ green_a[:nocc_a,:]
    c2_ggg_bbb = (greenp_b @ c2g_b.T) @ green_b[:nocc_b,:]
    c2_ggg_bab = (greenp_b @ c2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract("pq,pq->", h1_a, 4*c2_ggg_aaa + c2_ggg_aba, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, 4*c2_ggg_bbb + c2_ggg_bab, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # two body double excitations
    e2_2_1 = o2 * e2_0

    lc2g_a = oe.contract("gpq,pq->g", chol_a, 8 * c2_ggg_aaa + 2 * c2_ggg_aba, backend="jax")
    lc2g_b = oe.contract("gpq,pq->g", chol_b, 8 * c2_ggg_bbb + 2 * c2_ggg_bab, backend="jax")
    e2_2_2_1 = -((lc2g_a + lc2g_b) @ (trgl_a + trgl_b)) / 2.0

    def scanned_fun(carry, x):
        chol_a_i, chol_b_i, gl_a_i, gl_b_i = x
        lc2_ggg_a_i = oe.contract("pr,qr->pq", chol_a_i, 8 * c2_ggg_aaa + 2 * c2_ggg_aba, backend="jax")
        lc2_ggg_b_i = oe.contract("pr,qr->pq", chol_b_i, 8 * c2_ggg_bbb + 2 * c2_ggg_bab, backend="jax")
        carry[0] += (oe.contract("pq,pq->", gl_a_i, lc2_ggg_a_i, backend="jax")
                     + oe.contract("pq,pq->", gl_b_i, lc2_ggg_b_i, backend="jax")) / 2
        glgp_a_i = oe.contract("iq,qa->ia", gl_a_i[:nocc_a,:], greenp_a, backend="jax")
        glgp_b_i = oe.contract("iq,qa->ia", gl_b_i[:nocc_b,:], greenp_b, backend="jax")
        l2c2_aa = 0.5 * oe.contract("ia,jb,iajb->", glgp_a_i, glgp_a_i, c2_aa, backend="jax")
        l2c2_bb = 0.5 * oe.contract("ia,jb,iajb->", glgp_b_i, glgp_b_i, c2_bb, backend="jax")
        l2c2_ab = oe.contract("ia,jb,iajb->", glgp_a_i, glgp_b_i, c2_ab, backend="jax")
        carry[1] += l2c2_aa + l2c2_bb + l2c2_ab
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0, 0.0], (chol_a, chol_b, gl_a, gl_b))
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <C2 psi|h2|walker>/<psi|walker>

    energy = h0 + (e1_0 + e2_0 + e1_2 + e2_2) / (1 + o2)
    return overlap, energy

In [75]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
slater_up = jnp.array(np.random.rand(*walker_up.shape) + 1j*np.random.rand(*walker_up.shape)) * 0.5
slater_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j*np.random.rand(*walker_dn.shape)) * 0.5
wave_data['mo_ta'] = slater_up
wave_data['mo_tb'] = slater_dn

norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci2 = [wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']] 
olp1, energy1 = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci2)
print(olp1, energy1)
print(olp2, energy2)

(0.4693251761161401+0.1389904858527459j) (-0.4775885034613132-0.14297453250648984j)
(0.4693251761161401+0.1389904858527459j) (-0.4775885034613131-0.14297453250648978j)


In [91]:
@partial(jit, static_argnums=0)
def _calc_energy_cisd(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
    ci1, ci2,
) -> complex:
    nocc_a = self.nelec[0]
    nocc_b = self.nelec[1]
    c1_a, c1_b = ci1
    c2_aa, c2_ab, c2_bb = ci2
    c1_a = c1_a.conj() / 2
    c1_b = c1_b.conj() / 2
    c2_aa = c2_aa.conj()
    c2_ab = c2_ab.conj()
    c2_bb = c2_bb.conj()
    
    slater_up, slater_dn = wave_data['mo_ta'], wave_data['mo_tb']
    h0 = ham_data["h0"]
    h1_a = ham_data["h1"][0]
    h1_b = ham_data["h1"][1]
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)

    # full green's function G_pq
    green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn)
    greenov_a = green_a[:nocc_a, nocc_a:]
    greenov_b = green_b[:nocc_b, nocc_b:]
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    ################## overlaps #########################
    o0 = _walker_olp(walker_up, walker_dn, slater_up, slater_dn)
    o1 = oe.contract("ia,ia->", c1_a, greenov_a, backend="jax") \
        + oe.contract("ia,ia->", c1_b, greenov_b, backend="jax")
    o2 = 0.5 * oe.contract("iajb,ia,jb->", c2_aa, greenov_a, greenov_a, backend="jax") \
        + 0.5 * oe.contract("iajb,ia,jb->", c2_bb, greenov_b, greenov_b, backend="jax") \
        + oe.contract("iajb,ia,jb->", c2_ab, greenov_a, greenov_b, backend="jax")
    overlap =  (1.0 + o1 + o2) * o0

    ################## ref ###############################
    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    e1_0 = hg_a + hg_b # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>

    # two-body 
    gla = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
    glb = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
    trgla = oe.contract('gpp->g', gla, backend="jax")
    trglb = oe.contract('gpp->g', glb, backend="jax")
    e2_0_1 = 0.5 * jnp.sum((trgla + trglb)**2)
    e2_0_2 = - 0.5 * (oe.contract('gpq,gqp->', gla, gla, backend="jax")
                      + oe.contract('gpq,gqp->', glb, glb, backend="jax"))
    e2_0 = e2_0_1 + e2_0_2
    ########################################################

    # one body single excitations  <psi|T1 h1|walker>/<psi|HF|walker>
    e1_1_1 = o1 * e1_0

    gpc1_a = oe.contract("pa,ia->pi", greenp_a, c1_a, backend="jax") # greenp_a @ t1_a.T
    gpc1_b = oe.contract("pa,ia->pi", greenp_b, c1_b, backend="jax")
    c1_green_a = oe.contract("pi,iq->pq", gpc1_a, green_a[:nocc_a,:], backend="jax")
    c1_green_b = oe.contract("pi,iq->pq", gpc1_b, green_b[:nocc_b,:], backend="jax") # gpt1_b @ green_b
    e1_1_2 = -(oe.contract("pq,pq->", h1_a, c1_green_a, backend="jax")
               + oe.contract("pq,pq->", h1_b, c1_green_b, backend="jax"))
    
    e1_1 = e1_1_1 + e1_1_2 # <HF|T1 h1|walker>/<HF|walker>

    # one body double excitations  <psi|T2 h1|walker>/<psi|HF|walker>
    c2g_aa_a = oe.contract("iajb,ia->jb", c2_aa, greenov_a, backend="jax") / 4
    c2g_bb_b = oe.contract("iajb,ia->jb", c2_bb, greenov_b, backend="jax") / 4
    c2g_ab_a = oe.contract("iajb,jb->ia", c2_ab, greenov_b, backend="jax")
    c2g_ab_b = oe.contract("iajb,ia->jb", c2_ab, greenov_a, backend="jax")

    e1_2_1 = o2 * e1_0
    
    c2_ggg_aaa = (greenp_a @ c2g_aa_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    c2_ggg_aba = (greenp_a @ c2g_ab_a.T) @ green_a[:nocc_a,:]
    c2_ggg_bbb = (greenp_b @ c2g_bb_b.T) @ green_b[:nocc_b,:]
    c2_ggg_bab = (greenp_b @ c2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract("pq,pq->", h1_a, 4 * c2_ggg_aaa + c2_ggg_aba, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, 4 * c2_ggg_bbb + c2_ggg_bab, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <psi|T2 h1|walker>/<psi|walker>

    # two body single excitations <psi|T1 h2|walker>/<psi|walker>
    e2_1_1 = o1 * e2_0

    # c_ia Gp_pa G_ir L_pr G_qs L_qs
    lc1g_a = oe.contract("gpq,pq->g", chol_a, c1_green_a, backend="jax")
    lc1g_b = oe.contract("gpq,pq->g", chol_b, c1_green_b, backend="jax")
    e2_1_2 = -((lc1g_a + lc1g_b) @ (trgla + trglb))

    # t_ia Gp_pa G_qr G_is L_pr L_qs
    c1gp_a = oe.contract("ia,pa->ip", c1_a, greenp_a, backend="jax") # t_ia Gp_pa 
    c1gp_b = oe.contract("ia,pa->ip", c1_b, greenp_b, backend="jax")
    glgpc1_a = jnp.einsum("gpq,iq->gpi", gla, c1gp_a, optimize="optimal") # t_ia Gp_pa G_qr L_pr
    glgpc1_b = jnp.einsum("gpq,iq->gpi", glb, c1gp_b, optimize="optimal")
    e2_1_3 = jnp.einsum("gpi,gip->", glgpc1_a, gla[:,:nocc_a,:], optimize="optimal") \
            + jnp.einsum("gpi,gip->", glgpc1_b, glb[:,:nocc_b,:], optimize="optimal") # t_ia Gp_pa L_pr G_qr L_qs G_is
    
    e2_1 = e2_1_1 + e2_1_2 + e2_1_3 # <psi|ci1 h2|walker> / <psi|walker>

    # two body double excitations <psi|T2 h2|walker>/<psi|walker>
    e2_2_1 = o2 * e2_0

    lc2g_a = oe.contract("gpq,pq->g", chol_a, 8*c2_ggg_aaa + 2*c2_ggg_aba, backend="jax")
    lc2g_b = oe.contract("gpq,pq->g", chol_b, 8*c2_ggg_bbb + 2*c2_ggg_bab, backend="jax")
    e2_2_2_1 = -((lc2g_a + lc2g_b) @ (trgla + trglb)) / 2.0

    def scanned_fun(carry, x):
        chol_a_i, chol_b_i, gl_a_i, gl_b_i = x
        lc2_ggg_a_i = oe.contract("pr,qr->pq", chol_a_i, 8*c2_ggg_aaa + 2*c2_ggg_aba, backend="jax")
        lc2_ggg_b_i = oe.contract("pr,qr->pq", chol_b_i, 8*c2_ggg_bbb + 2*c2_ggg_bab, backend="jax")
        carry[0] += 0.5 * (oe.contract("pq,pq->", gl_a_i, lc2_ggg_a_i, backend="jax")
                           + oe.contract("pq,pq->", gl_b_i, lc2_ggg_b_i, backend="jax"))
        glgp_a_i = oe.contract("iq,qa->ia", gl_a_i[:nocc_a,:], greenp_a, backend="jax")
        glgp_b_i = oe.contract("iq,qa->ia", gl_b_i[:nocc_b,:], greenp_b, backend="jax")
        l2c2_aa = oe.contract("ia,jb,iajb->", 
                                glgp_a_i.astype(jnp.complex64), 
                                glgp_a_i.astype(jnp.complex64),
                                c2_aa.astype(jnp.complex64), 
                                backend="jax") / 2
        l2c2_bb = oe.contract("ia,jb,iajb->", 
                                glgp_b_i.astype(jnp.complex64), 
                                glgp_b_i.astype(jnp.complex64), 
                                c2_bb.astype(jnp.complex64), 
                                backend="jax") / 2
        l2c2_ab = oe.contract("ia,jb,iajb->", 
                                glgp_a_i.astype(jnp.complex64), 
                                glgp_b_i.astype(jnp.complex64), 
                                c2_ab.astype(jnp.complex64), 
                                backend="jax")
        carry[1] += l2c2_aa + l2c2_ab + l2c2_bb
        return carry, 0.0

    [e2_2_2_2, e2_2_3], _ = lax.scan(scanned_fun, [0.0, 0.0], (chol_a, chol_b, gla, glb))

    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <psi|T2 h2|walker>/<psi|walker>

    energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1 + o1 + o2)
    return overlap, energy

In [97]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
# ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci1a = jnp.array(np.random.rand(nocca, nvira) + 1j*np.random.rand(nocca, nvira))
ci1b = jnp.array(np.random.rand(noccb, nvirb) + 1j*np.random.rand(noccb, nvirb))
ci2aa = jnp.einsum('ia,jb->iajb', ci1a, ci1a)
ci2ab = jnp.einsum('ia,jb->iajb', ci1a, ci1b)
ci2bb = jnp.einsum('ia,jb->iajb', ci1b, ci1b)
ci1 = [ci1a, ci1b]
# ciaa = jnp.array(np.random.rand(*wave_data['t2aa'].shape) + 1j*np.random.rand(*wave_data['t2aa'].shape)) * 0.5
ci2 = [wave_data['t2aa'], wave_data['t2ab'], wave_data['t2bb']]
olp1, energy1 = _calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp1, energy1)
# olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
# print(olp2, energy2)
olp3, energy3 = _calc_energy_cisd(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
print(olp3, energy3)

(0.8751727603901351-0.548150123657969j) (-0.2075504047784511-0.2798992836208454j)
(0.8751727603901351-0.5481501236579691j) (-0.20755040449070955-0.27989928402336406j)


In [ ]:
# @partial(jit, static_argnums=0)
# def _calc_energy_cisd_dc(
#     trial, 
#     walker_up: jax.Array,
#     walker_dn: jax.Array,
#     ham_data: dict, 
#     wave_data: dict,
#     ci1,
#     ):

#     '''
#     Disconnected Doubles!!! c_iajb = c_ia c_jb
#     A local energy evaluator for <(1+C1+C2)psi|H|walker> / <(1+C1+C2)psi|walker>
#     all operators and the walkers and psi are in the same basis (normally MO)
#     |psi> is not necesarily diagonal
    
#     all green's function and the chol and ci coeff are as their original definition
#     no half rotation performed
#     '''
#     norb = trial.norb
#     nocc_a, nocc_b = trial.nelec
#     h0  = ham_data['h0']
#     h1_a, h1_b = ham_data["h1"]
#     slater_up, slater_dn = wave_data["mo_ta"], wave_data["mo_tb"]
#     chol_a = ham_data["chol"][0].reshape(-1, trial.norb, trial.norb)
#     chol_b = ham_data["chol"][1].reshape(-1, trial.norb, trial.norb)
#     green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn) # full green
#     greenov_a = green_a[:nocc_a, nocc_a:]
#     greenov_b = green_b[:nocc_b, nocc_b:]
#     greenp_a = (green_a - jnp.eye(trial.norb))[:, nocc_a:]
#     greenp_b = (green_b - jnp.eye(trial.norb))[:, nocc_b:]
    
#     # applied to the bra
#     c1_a, c1_b = ci1
#     c1_a = c1_a.conj()
#     c1_b = c1_b.conj()

#     ######################## universal terms #########################
#     c1g_a = oe.contract("ia,ja->ij", c1_a, greenov_a, backend="jax")
#     c1g_b = oe.contract("ia,ja->ij", c1_b, greenov_b, backend="jax")
#     c1gp_a = oe.contract("ia,pa->ip", c1_a, greenp_a, backend="jax")
#     c1gp_b = oe.contract("ia,pa->ip", c1_b, greenp_b, backend="jax")
    
#     ########################## overlap terms #########################
#     o0 = _walker_olp(walker_up, walker_dn, slater_up, slater_dn)
#     o1_a = oe.contract("ii->", c1g_a, backend="jax") 
#     o1_b = oe.contract("ii->", c1g_b, backend="jax")
#     o1 = o1_a + o1_b
#     o2 = 0.5 * o1**2
#     overlap = (1.0 + o1 + o2) * o0
    
#     ########################### ref energy ############################
#     hg_a = oe.contract("pr,qr->pq", h1_a, green_a, backend="jax")
#     hg_b = oe.contract("pr,qr->pq", h1_b, green_b, backend="jax")
#     trhg_a = oe.contract("pp->", hg_a, backend="jax")
#     trhg_b = oe.contract("pp->", hg_b, backend="jax")
#     e1_0 = trhg_a + trhg_b
 
#     gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
#     gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
#     trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
#     trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
#     e2_0_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
#     e2_0_2 = -(oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
#                + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
#     e2_0 = e2_0_1 + e2_0_2

#     ############################ ci terms #############################

#     ###### one-body single excitations ######
#     e1_1_1 = o1 * e1_0

#     c1gpg_a = oe.contract("ip,iq->pq", c1gp_a, green_a[:nocc_a,:], backend="jax")
#     c1gpg_b = oe.contract("ip,iq->pq", c1gp_b, green_b[:nocc_b,:], backend="jax")
#     e1_1_2 = -(oe.contract("pq,pq->", c1gpg_a, h1_a, backend="jax")
#                + oe.contract("pq,pq->", c1gpg_b, h1_b, backend="jax"))
    
#     e1_1 = e1_1_1 + e1_1_2 # <C1 psi|h1|walker>/<psi|walker>

#     ###### one-body double excitations ######
#     e1_2_1 = o2 * e1_0

#     c2ggg_aaa = o1_a * c1gpg_a / 4 # cA_ia GA_ia cA_jb GpA_pb GA_jq
#     c2ggg_aba = o1_b * c1gpg_a # cB_ia GB_ia  cA_jb GpA_pb  GA_jq
#     c2ggg_bbb = o1_b * c1gpg_b / 4
#     c2ggg_bab = o1_a * c1gpg_b
#     e1_2_2_a = -oe.contract("pq,pq->", 4*c2ggg_aaa + c2ggg_aba, h1_a, backend="jax")
#     e1_2_2_b = -oe.contract("pq,pq->", 4*c2ggg_bbb + c2ggg_bab, h1_b, backend="jax")
#     e1_2_2 = e1_2_2_a + e1_2_2_b
#     e1_2 = e1_2_1 + e1_2_2  # <C2 psi|h1|walker>/<psi|walker>

#     ###### two-body single excitations ######
#     e2_1_1 = o1 * e2_0

#     # c_ia Gp_pa G_ir L_pr G_qs L_qs
#     lc1g_a = oe.contract("gpq,pq->g", chol_a, c1gpg_a, backend="jax")
#     lc1g_b = oe.contract("gpq,pq->g", chol_b, c1gpg_b, backend="jax")
#     e2_1_2 = -((lc1g_a + lc1g_b) @ (trgl_a + trgl_b))

#     glc1gp_a = jnp.einsum("gpq,iq->gpi", gl_a, c1gp_a, optimize="optimal") # t_ia Gp_qa G_pr L_qr 
#     glc1gp_b = jnp.einsum("gpq,iq->gpi", gl_b, c1gp_b, optimize="optimal")
#     e2_1_3 = jnp.einsum("gpi,gip->", glc1gp_a, gl_a[:,:nocc_a,:], optimize="optimal") \
#             + jnp.einsum("gpi,gip->", glc1gp_b, gl_b[:,:nocc_b,:], optimize="optimal") # t_ia Gp_pa G_qr L_pr G_is L_qs
    
#     e2_1 = e2_1_1 + e2_1_2 + e2_1_3 # <C1 psi|h2|walker> / <psi|walker>

#     ###### two-body double excitations ######
#     e2_2_1 = o2 * e2_0

#     lt2g_a = oe.contract("gpq,pq->g", chol_a, 8*c2ggg_aaa + 2*c2ggg_aba, backend="jax")
#     lt2g_b = oe.contract("gpq,pq->g", chol_b, 8*c2ggg_bbb + 2*c2ggg_bab, backend="jax")
#     e2_2_2_1 = -((lt2g_a + lt2g_b) @ (trgl_a + trgl_b)) / 2

#     lc2ggg_a = oe.contract("gpr,qr->gpq", chol_a, 8*c2ggg_aaa + 2*c2ggg_aba, backend="jax")
#     lc2ggg_b = oe.contract("gpr,qr->gpq", chol_b, 8*c2ggg_bbb + 2*c2ggg_bab, backend="jax")
#     e2_2_2_2 = (oe.contract("gpq,gpq->", gl_a, lc2ggg_a, backend="jax")
#                 + oe.contract("gpq,gpq->", gl_b, lc2ggg_b, backend="jax")) / 2
#     glgp_a = oe.contract("giq,qa->gia", gl_a[:,:nocc_a,:], greenp_a, backend="jax")
#     glgp_b = oe.contract("giq,qa->gia", gl_b[:,:nocc_b,:], greenp_b, backend="jax")
#     l2t2_a = oe.contract("gia,ia->g", glgp_a, c1_a, backend="jax")
#     l2t2_b = oe.contract("gia,ia->g", glgp_b, c1_b, backend="jax")
#     e2_2_3 = jnp.sum((l2t2_a + l2t2_b)**2) / 2

#     e2_2_2 = e2_2_2_1 + e2_2_2_2
#     e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <C2 psi|h2|walker>/<psi|walker>

#     energy = h0 + (e1_0 + e2_0 + e1_1 + e2_1 + e1_2 + e2_2) / (1 + o1 + o2)

#     return overlap, energy

In [ ]:
walker_up = prop_data['walkers'][0][0]
walker_dn = prop_data['walkers'][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
walker_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
slater_up = jnp.array(np.random.rand(*walker_up.shape) + 1j * np.random.rand(*walker_up.shape)) * 0.5
slater_dn = jnp.array(np.random.rand(*walker_dn.shape) + 1j * np.random.rand(*walker_dn.shape)) * 0.5
wave_data['mo_ta'] = slater_up
wave_data['mo_tb'] = slater_dn
norb = trial.norb
nocca, noccb = trial.nelec
nvira, nvirb = (norb-nocca, norb-noccb)
# ci1 = [jnp.zeros((nocca, nvira)), jnp.zeros((noccb, nvirb))]
ci1a = jnp.array(np.random.rand(nocca, nvira) + 1j*np.random.rand(nocca, nvira)) * 0.5
ci1b = jnp.array(np.random.rand(noccb, nvirb) + 1j*np.random.rand(noccb, nvirb)) * 0.5
ci2aa = jnp.einsum('ia,jb->iajb', ci1a, ci1a)
ci2ab = jnp.einsum('ia,jb->iajb', ci1a, ci1b)
ci2bb = jnp.einsum('ia,jb->iajb', ci1b, ci1b)
ci1 = [ci1a, ci1b]
# ciaa = jnp.array(np.random.rand(*wave_data['t2aa'].shape) + 1j*np.random.rand(*wave_data['t2aa'].shape)) * 0.5
ci2 = [ci2aa, ci2ab, ci2bb]
olp1, energy1 = _calc_energy_slater(trial, walker_up, walker_dn, slater_up, slater_dn, ham_data, wave_data)
print(olp1, energy1)
# olp2, energy2 = _calc_energy_cid(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
# print(olp2, energy2)
# olp3, energy3 = _calc_energy_cisd(trial, walker_up, walker_dn, ham_data, wave_data, ci1, ci2)
# print(olp3, energy3)
olp4, energy4 = _calc_energy_cisd_dc(trial, walker_up, walker_dn, ham_data, wave_data, ci1)
print(olp4, energy4)

(0.3090011489922681-0.11982103000821652j) (0.0688251638440352+0.24555066933502867j)
(0.3090011489922681-0.11982103000821652j) (0.06882516310995715+0.24555066750832955j)
(0.3090011489922681-0.11982103000821655j) (0.06882516384403545+0.245550669335029j)


In [ ]:
# @partial(jit, static_argnums=0)
# def _calc_energy_slater(
#     self,
#     walker_up: jax.Array,
#     walker_dn: jax.Array,
#     slater_up: jax.Array,
#     slater_dn: jax.Array,
#     ham_data: dict,
#     ) -> jax.Array:
    
#     norb = trial.norb
#     nocc_a, nocc_b = trial.nelec
#     h0  = ham_data['h0']
#     h1_a, h1_b = ham_data["h1"]
#     chol_a = ham_data["chol"][0].reshape(-1, trial.norb, trial.norb)
#     chol_b = ham_data["chol"][1].reshape(-1, trial.norb, trial.norb)
#     green_a, green_b = _green(walker_up, walker_dn, slater_up, slater_dn)
    
#     hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
#     hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
#     e1 = hg_a + hg_b
 
#     gl_a = oe.contract("pr,gqr->gpq", green_a, chol_a, backend="jax")
#     gl_b = oe.contract("pr,gqr->gpq", green_b, chol_b, backend="jax")
#     trgl_a = oe.contract('gpp->g', gl_a, backend="jax")
#     trgl_b = oe.contract('gpp->g', gl_b, backend="jax")
#     e2_1 = jnp.sum((trgl_a + trgl_b)**2) / 2
#     e2_2 = -(oe.contract('gpq,gqp->', gl_a, gl_a, backend="jax")
#                + oe.contract('gpq,gqp->', gl_b, gl_b, backend="jax")) / 2
#     e2 = e2_1 + e2_2

#     overlap = _walker_olp(walker_up, walker_dn, slater_up, slater_dn)
#     energy = h0 + e1 + e2

#     return overlap, energy